In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from Log_Extractor import LogExtractor
from Fast_Log_Ext import Fast_LogExtractor
import os

In [3]:
# ==========================================
# ⚙️ 1. 설정 및 매개변수 (여기만 바꾸면 14개 펌프 무한 확장 가능!)
# ==========================================
PUMP_ID = 'P1' # 나중에 P2, P3 등으로 변경
TANK_ID = 'P1' # 나중에 T2, T3 등으로 변경
ROBOT_ID = 'RB1' # 나중에 R2, R3 등으로 변경
MODEL_SAVE_DIR = './Ana_models' # 모델과 스케일러를 저장할 폴더
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

pid = PUMP_ID
tank_id = TANK_ID
robot_id = ROBOT_ID

In [4]:
def prepare_pump_features(raw_df, pump_id, tank_id, robot_id):
    """
    raw_df를 받아 해당 펌프(pump_id)의 피처를 생성하여 model_df를 반환합니다.
    """
    df = raw_df.ffill().fillna(0).copy()
    df_reset = df.reset_index() 
    
    # 동적 태그 이름 생성 (P1, P2 등 자동 대응)
    tag_SV = f'g_s_SV_{pump_id}'
    tag_Ana = f'Ana_Out_{pump_id}'
    tag_PT = f'Scale_Out___PT_{pump_id}'
    tag_FT = f'Scale_Out___FT_{pump_id}'
    tag_Temp = f'TK_Temp_PV_{tank_id}'

    # [신규] 완벽한 타이밍을 위한 하드 트리거 태그
    tag_BuildUp = f'Pump_BuildUp_{pump_id}' 
    tag_Wagon = f'{robot_id}_Robot_Num'

    # ==========================================
    # ✂️ 1. 완벽한 칼단발 블록(Block) 나누기
    # ==========================================
    # 대차 번호가 바뀌거나, 펌프 가동 상태(ON/OFF)가 바뀔 때마다 새로운 블록 ID를 부여합니다.
    df_reset['Wagon_Changed'] = df_reset[tag_Wagon].diff() != 0
    df_reset['BuildUp_Changed'] = df_reset[tag_BuildUp].diff() != 0
    df_reset['Wagon_Block_ID'] = (df_reset['Wagon_Changed'] | df_reset['BuildUp_Changed']).cumsum()

    # ==========================================
    # 🎯 2. 진짜 샷 추출 (조건부 필터링 싹 다 폐기)
    # ==========================================
    # BuildUp이 1(가동 중)인 데이터만 남깁니다. 이것이 진짜 샷입니다.
    df_shots = df_reset[df_reset[tag_BuildUp] == 1].copy()

    # ==========================================
    # 🔄 3. 이전 SV 값(Prev_SV) 구하기
    # ==========================================
    # 각 유효 블록별로 최대 SV를 구한 뒤, 이전 샷의 SV를 1칸 당겨옵니다.
    block_summary = df_shots.groupby('Wagon_Block_ID').agg(
        Max_SV=(tag_SV, 'max')
    )
    block_summary['Prev_SV'] = block_summary['Max_SV'].shift(1)

    # 원본 df_shots에 Prev_SV 정보 맵핑
    df_shots = df_shots.merge(block_summary[['Prev_SV']], left_on='Wagon_Block_ID', right_index=True, how='left')
    df_shots = df_shots.sort_index()

    # ==========================================
    # 📈 4. 실시간 피처 엔지니어링 (기존 로직 완벽 유지)
    # ==========================================
    model_df = df_shots.copy()
    model_df['Tick_Index'] = model_df.groupby('Wagon_Block_ID').cumcount()
    model_df['Phase_Start'] = np.where(model_df['Tick_Index'] < 2, 1.0, 0.0)
    model_df['Phase_Steady'] = np.where(model_df['Tick_Index'] >= 2, 1.0, 0.0)

    model_df['Instant_FT_Error'] = model_df[tag_SV] - model_df[tag_FT]
    model_df['Instant_FT_Error_Rate'] = np.where(
        model_df[tag_SV] > 0, 
        (model_df['Instant_FT_Error'] / model_df[tag_SV]) * 100, 
        0
    )
    model_df['Cum_FT_Error'] = model_df.groupby('Wagon_Block_ID')['Instant_FT_Error'].cumsum()

    model_df['Prev_SV_Diff'] = model_df[tag_SV] - model_df['Prev_SV'].fillna(0)
    model_df['Rolling_PT_Max_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].rolling(window=3, min_periods=1).max().reset_index(0, drop=True)
    model_df['Rolling_PT_Diff_3'] = model_df.groupby('Wagon_Block_ID')[tag_PT].diff(periods=2).fillna(0)

    model_df['Instant_SV_Diff'] = model_df[tag_SV].diff().fillna(0)
    model_df['Phase_Transition'] = np.where(model_df['Instant_SV_Diff'] != 0, 1.0, 0.0)

    # ==========================================
    # 🧹 5. 최종 컬럼 정리
    # ==========================================
    feature_cols = [
        tag_SV, 'Prev_SV', 'Prev_SV_Diff', 
        tag_Ana, tag_Temp,               
        tag_PT, 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
        tag_FT, 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
        'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
    ]
    
    # 결측치 최종 정리 (Shift나 Rolling에서 발생한 NaN 처리)
    model_df = model_df.fillna(0)
    
    return model_df[feature_cols], feature_cols

print("✅ 동적 피처 생성 함수 준비 완료! (하드 트리거 기반)")

✅ 동적 피처 생성 함수 준비 완료! (하드 트리거 기반)


In [5]:
# 한글 폰트 설정
plt.rc('font', family='AppleGothic') 
plt.rcParams['axes.unicode_minus'] = False

# 1. 추출기 가동 (어제 하루치 데이터만 가져오기)
# extractor는 이미 위에서 선언했다고 가정합니다.
target_tags = [
    f'Pump_BuildUp_{PUMP_ID}' , f'{ROBOT_ID}_Robot_Num', 
    f'g_s_SV_{PUMP_ID}', f'Ana_Out_{PUMP_ID}', 
    f'Scale_Out___PT_{PUMP_ID}', f'Scale_Out___FT_{PUMP_ID}', 
    f'TK_Temp_PV_{TANK_ID}', f'TK_Level_PV_{TANK_ID}'
]
extractor = LogExtractor()
fast_ex = Fast_LogExtractor()

🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!
🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!


In [6]:
extract = LogExtractor()
start_time="2026-04-13T01:00:00Z"
end_time="2026-04-14T15:00:00Z"

raw_df = extract.get_data(start_time=start_time, end_time=end_time, target_tags=target_tags)
train_df, feature_cols = prepare_pump_features(raw_df, pid, tank_id, robot_id)
print(f"✅ {pid} 데이터 준비 완료! 총 {len(train_df)}틱, 피처 수: {len(feature_cols)}")

🔌 [Extractor] InfluxDB 분석용 추출기 연결 완료!
🚀 전체 구간 데이터 추출 시작: 2026-04-13T01:00:00+00:00 ~ 2026-04-14T15:00:00+00:00
📦 [Chunk] 2026-04-13T01:00:00Z ~ 2026-04-13T07:00:00Z 요청 중...
📦 [Chunk] 2026-04-13T07:00:00Z ~ 2026-04-13T13:00:00Z 요청 중...
📦 [Chunk] 2026-04-13T13:00:00Z ~ 2026-04-13T19:00:00Z 요청 중...
📦 [Chunk] 2026-04-13T19:00:00Z ~ 2026-04-14T01:00:00Z 요청 중...
📦 [Chunk] 2026-04-14T01:00:00Z ~ 2026-04-14T07:00:00Z 요청 중...
📦 [Chunk] 2026-04-14T07:00:00Z ~ 2026-04-14T13:00:00Z 요청 중...
📦 [Chunk] 2026-04-14T13:00:00Z ~ 2026-04-14T15:00:00Z 요청 중...
✅ 전체 데이터 통합 완료! 총 57776행 확보.
✅ P1 데이터 준비 완료! 총 20869틱, 피처 수: 14


In [7]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from Ana_Preprocess import VirtualSensorPreprocessor

print("🏋️‍♂️ 가상 센서 (컨닝 방지 버전) 재학습 시작...")

# 1. 아까 정의한 '순수 물리 피처' 7개
pure_physical_features = [
    f'Scale_Out___PT_{pid}', 
    'Rolling_PT_Max_3', 
    'Rolling_PT_Diff_3', 
    f'Scale_Out___FT_{pid}', 
    f'TK_Temp_PV_{tank_id}',
    'Phase_Start', 
    'Phase_Steady'
]

# X: 순수 물리 피처 7개만 쏙 뽑기
X_train = train_df[pure_physical_features]
# y: 정답지 (Ana_Out)
y_train = train_df[f'Ana_Out_{pid}']

print(f"📊 학습 데이터 준비 완료: {X_train.shape} (7개 피처만 사용!)")

# 3. 모델 학습 (Random Forest)
virtual_sensor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
virtual_sensor.fit(X_train, y_train)

# 4. 저장 (기존 구형 모델 덮어쓰기)
joblib.dump(virtual_sensor, f'Ana_models/virtual_sensor_{pid}.pkl')
print("✅ 순수 물리 가상 센서 저장 완료! 이제 아까 그 테스트 코드를 다시 돌려보세요!")

🏋️‍♂️ 가상 센서 (컨닝 방지 버전) 재학습 시작...
📊 학습 데이터 준비 완료: (20869, 7) (7개 피처만 사용!)
✅ 순수 물리 가상 센서 저장 완료! 이제 아까 그 테스트 코드를 다시 돌려보세요!


In [8]:
raw_valid = extract.get_data(start_time="-1d", end_time="now()", target_tags=target_tags)

🚀 전체 구간 데이터 추출 시작: 2026-04-19T06:31:56.279820+00:00 ~ 2026-04-20T06:31:56.279832+00:00
📦 [Chunk] 2026-04-19T06:31:56Z ~ 2026-04-19T12:31:56Z 요청 중...
📦 [Chunk] 2026-04-19T12:31:56Z ~ 2026-04-19T18:31:56Z 요청 중...
📦 [Chunk] 2026-04-19T18:31:56Z ~ 2026-04-20T00:31:56Z 요청 중...
📦 [Chunk] 2026-04-20T00:31:56Z ~ 2026-04-20T06:31:56Z 요청 중...
📦 [Chunk] 2026-04-20T06:31:56Z ~ 2026-04-20T06:31:56Z 요청 중...
❌ [Chunk 에러] 2026-04-20T06:31:56Z 구간 실패: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'Vary': 'Accept-Encoding', 'X-Influxdb-Build': 'OSS', 'X-Influxdb-Version': 'v2.7.12', 'X-Platform-Error-Code': 'invalid', 'Date': 'Mon, 20 Apr 2026 06:33:22 GMT', 'Transfer-Encoding': 'chunked'})
HTTP response body: b'{"code":"invalid","message":"error in building plan while starting program: cannot query an empty range"}'

✅ 전체 데이터 통합 완료! 총 24280행 확보.


In [9]:
import joblib
from Ana_Preprocess import VirtualSensorPreprocessor

print("🚀 가상 센서 (Virtual Sensor) 단독 테스트 시작...\n")

# 1. 학습된 가상 센서(Random Forest) 로드
# (앞서 Ana_Out을 정답지(y)로 두고 학습한 모델이 있다고 가정합니다)
try:
    virtual_sensor = joblib.load(f'Ana_models/virtual_sensor_{pid}.pkl')
    print("✅ 가상 센서 모델 로드 완료!")
except FileNotFoundError:
    print("⚠️ 모델 파일이 없습니다. 코드가 정상 작동하는지 흐름만 테스트합니다.")
    virtual_sensor = None

# 2. 피처 컬럼 세팅 (Ana_Out_P1 제외!)
all_features = [
    f'g_s_SV_{pid}', f'Prev_SV', f'Prev_SV_Diff', f'Ana_Out_{pid}', f'TK_Temp_PV_{tank_id}',               
    f'Scale_Out___PT_{pid}', 'Rolling_PT_Max_3', 'Rolling_PT_Diff_3', 
    f'Scale_Out___FT_{pid}', 'Instant_FT_Error_Rate', 'Cum_FT_Error',  
    'Phase_Start', 'Phase_Steady', 'Phase_Transition'             
]

# 🕵️‍♂️ [핵심] 가상 센서용 순수 물리 피처 (컨닝 방지!)
pure_physical_features = [
    f'Scale_Out___PT_{pid}', 
    'Rolling_PT_Max_3', 
    'Rolling_PT_Diff_3', 
    f'Scale_Out___FT_{pid}', 
    f'TK_Temp_PV_{tank_id}',
    'Phase_Start', 
    'Phase_Steady'
]

# 전처리기 가동
vs_preprocessor = VirtualSensorPreprocessor(feature_cols=all_features)

# 3. 실시간 스트리밍 시뮬레이션 루프
records = raw_valid.to_dict('records')
loss_alerts = []

for i, live_row in enumerate(records):
    # 전처리기에 1틱 입력
    df_processed, meta_info = vs_preprocessor.process_raw_tick(live_row)
    
    # 가동 중이 아니면(BuildUp == 0) 패스
    if df_processed is None:
        continue
        
    timestamp = live_row.get('Time', 'Unknown Time')

    # PLC가 명령한 원래 Ana_Out 값 추출
    commanded_ana = df_processed.iloc[0]['Ana_Out_P1']
    
    # 🚨 모델 입력 데이터는 '순수 물리 피처'만 칼같이 잘라냅니다!
    X_input = df_processed[pure_physical_features]
    
    if virtual_sensor:
        # 가상 센서 역산: "현재 물리적 결과(PT, FT 등)를 내려면 실제 전류가 얼마여야 하는가?"
        virtual_ana = virtual_sensor.predict(X_input)[0]
        print(f"[{timestamp}] [틱 {meta_info['Tick_Index']:02d}] 명령 Ana: {commanded_ana:.0f} | 가상 역산: {virtual_ana:.0f}")
        
        # 효율 손실률 계산 (%)
        if commanded_ana > 0:
            power_loss_rate = ((commanded_ana - virtual_ana) / commanded_ana) * 100
        else:
            power_loss_rate = 0.0
            
        # 15% 이상 힘이 증발했다면 경고 출력
        if power_loss_rate > 5.0:
            robot_num = meta_info['Robot_Num']
            tick = meta_info['Tick_Index']
            msg = f"🚨 [로봇 {robot_num}번 / {tick}틱] 전력 손실 감지! (명령: {commanded_ana:.0f} vs 가상 역산: {virtual_ana:.0f} | 손실률: {power_loss_rate:.1f}%)"
            print(msg)
            loss_alerts.append(msg)

print("\n" + "="*60)
print(f"✅ 가상 센서 테스트 완료! 총 {len(records)}틱 중 {len(loss_alerts)}회의 전력 손실 알람 발생.")
print("="*60)

🚀 가상 센서 (Virtual Sensor) 단독 테스트 시작...

✅ 가상 센서 모델 로드 완료!
[Unknown Time] [틱 00] 명령 Ana: 12242 | 가상 역산: 11577
🚨 [로봇 26.0번 / 0틱] 전력 손실 감지! (명령: 12242 vs 가상 역산: 11577 | 손실률: 5.4%)
[Unknown Time] [틱 01] 명령 Ana: 12242 | 가상 역산: 11811
[Unknown Time] [틱 00] 명령 Ana: 12242 | 가상 역산: 11813
[Unknown Time] [틱 01] 명령 Ana: 12242 | 가상 역산: 11807
[Unknown Time] [틱 00] 명령 Ana: 12242 | 가상 역산: 12022
[Unknown Time] [틱 01] 명령 Ana: 12242 | 가상 역산: 12017
[Unknown Time] [틱 02] 명령 Ana: 12242 | 가상 역산: 12017
[Unknown Time] [틱 00] 명령 Ana: 12242 | 가상 역산: 12018
[Unknown Time] [틱 01] 명령 Ana: 12242 | 가상 역산: 12017
[Unknown Time] [틱 02] 명령 Ana: 12242 | 가상 역산: 12017
[Unknown Time] [틱 03] 명령 Ana: 12242 | 가상 역산: 11813
[Unknown Time] [틱 00] 명령 Ana: 8938 | 가상 역산: 8942
[Unknown Time] [틱 01] 명령 Ana: 8938 | 가상 역산: 9008
[Unknown Time] [틱 00] 명령 Ana: 9464 | 가상 역산: 9465
[Unknown Time] [틱 01] 명령 Ana: 9464 | 가상 역산: 9464
[Unknown Time] [틱 00] 명령 Ana: 10894 | 가상 역산: 9136
🚨 [로봇 36.0번 / 0틱] 전력 손실 감지! (명령: 10894 vs 가상 역산: 9136 | 손실률: 16.1%)


In [10]:
fast_ex.save_to_csv(raw_valid, save_dir='./Ana_csv')

💾 [저장 완료] 분석용 CSV 파일이 생성되었습니다: ./Ana_csv/2026-04-20_153838_analysis.csv


In [13]:
raw_valid

,Pump_BuildUp_P1,RB1_Robot_Num,g_s_SV_P1,Ana_Out_P1,Scale_Out___PT_P1,Scale_Out___FT_P1,TK_Temp_PV_P1,TK_Level_PV_P1
Time,,,,,,,,
2026-04-14 10:30:42.651238,1.0,37.0,2680.0,10935.0,122.0,1795.0,NaN,NaN
2026-04-14 10:30:43.367209,1.0,37.0,2680.0,10935.0,127.0,2687.0,NaN,NaN
2026-04-14 10:30:45.145280,1.0,37.0,2680.0,10935.0,124.0,2687.0,NaN,NaN
2026-04-14 10:30:47.993071,1.0,37.0,2680.0,10935.0,126.0,2687.0,NaN,NaN
2026-04-14 10:30:50.928229,0.0,37.0,2680.0,5333.0,66.0,1534.0,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-04-15 10:30:29.358077,1.0,29.0,2340.0,9464.0,105.0,2340.0,21.0,58.0
2026-04-15 10:30:31.369717,1.0,29.0,2340.0,9464.0,108.0,2337.0,21.0,58.0
2026-04-15 10:30:33.368898,0.0,29.0,2340.0,5333.0,68.0,1548.0,21.0,58.0
